# Turtle Draw — Pipeline de Visão Computacional

Pega uma foto, isola o objeto escuro do fundo claro com **threshold**, detecta o contorno com **Sobel**, e gera segmentos horizontais que a tartaruga vai percorrer linha a linha — como uma impressora matricial.

Tudo está em um único arquivo: [`turtle_circle/pipeline.py`](turtle_circle/pipeline.py). Este notebook só chama as funções e mostra os resultados.

**Restrições atendidas:** `cv2` aparece em uma única linha (`cv2.imread`). Todo o resto é NumPy puro.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from turtle_circle import pipeline as p

plt.rcParams['figure.figsize'] = (10, 5)
IMG = 'images/dog.png'

## 1. Carregar e converter para cinza

Cada pixel da imagem original tem 3 valores (azul, verde, vermelho). A média desses 3 dá o tom de cinza. Não precisamos de informação de cor para fazer threshold de intensidade.

In [ ]:
img = p.carregar(IMG)   # cv2.imread devolve BGR (azul, verde, vermelho)
cinza = p.para_cinza(img)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
# matplotlib espera RGB, entao invertemos os canais com [:, :, ::-1]
ax[0].imshow(img[:, :, ::-1]); ax[0].set_title('Original'); ax[0].axis('off')
ax[1].imshow(cinza, cmap='gray'); ax[1].set_title('Cinza'); ax[1].axis('off')
plt.show()

## 2. Reduzir o tamanho

Cada linha da imagem reduzida vira uma passada da "impressora". A tartaruga é lenta, então não dá para usar centenas de linhas. 70 px de largura (→ 39 linhas) é um bom equilíbrio entre detalhe e tempo de desenho.

In [ ]:
pequena = p.redimensionar(cinza, 70)
# .shape devolve (altura, largura). A altura sai proporcional a largura
print(f'Antes: {cinza.shape} → Depois: {pequena.shape}')
plt.imshow(pequena, cmap='gray'); plt.title('Reduzida'); plt.axis('off'); plt.show()

## 3. Binarizar (threshold de intensidade)

**Observação:** o cachorro é visivelmente mais escuro que o fundo. Então `cinza < limiar` já isola a silhueta. Este passo é importante porque limpa o problema **antes** da detecção de bordas — se aplicássemos Sobel direto na foto original, a textura dos tijolos e do pelo geraria centenas de bordas espúrias.

In [ ]:
# A comparacao devolve um array booleano: True onde pixel < 0.45
mascara = p.binarizar_escuros(pequena, limiar=0.45)
print(f'Pixels na silhueta: {int(mascara.sum())}')
plt.imshow(mascara, cmap='gray_r'); plt.title('Máscara (preto = cachorro)'); plt.axis('off'); plt.show()

## 4. Detecção de bordas com Sobel

**Borda = mudança brusca de intensidade entre pixels vizinhos.** O filtro Sobel calcula a derivada da imagem em x e y usando dois kernels 3×3. Onde a derivada é grande, há uma borda.

A magnitude $\sqrt{G_x^2 + G_y^2}$ mede a "força" da borda em cada pixel. Como aplicamos sobre a máscara binária (silhueta limpa), o resultado destaca exatamente o contorno do cachorro.

In [ ]:
# Sobel faz aritmetica (multiplicacao por kernel), entao converte bool -> float
magnitude = p.sobel(mascara.astype(float))
# Binariza a magnitude: pixels com gradiente forte viram borda
bordas = magnitude > 0.5
print(f'Pixels de borda: {int(bordas.sum())}')

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(magnitude, cmap='magma'); ax[0].set_title('Magnitude (Sobel)'); ax[0].axis('off')
ax[1].imshow(bordas, cmap='gray_r'); ax[1].set_title('Bordas binarizadas'); ax[1].axis('off')
plt.show()

## 5. Varredura linha a linha

Para cada linha do mapa de bordas, percorremos os pixels da esquerda para a direita. Trechos contíguos de borda viram segmentos `[(i, j_início), (i, j_fim)]`. Trechos sem borda são pulados (a caneta sobe). Cada segmento é um "traço" para a tartaruga.

In [ ]:
h, w = bordas.shape
segmentos = p.varrer_linhas(bordas)
# Cada segmento sai em coordenadas (i,j) da imagem;
# mapear_turtlesim converte para (x,y) do canvas 11x11
tracos = [p.mapear_turtlesim(seg, h, w) for seg in segmentos]

total = sum(len(t) for t in tracos)
print(f'{len(tracos)} segmentos, {total} waypoints')

# Visualiza o caminho que a tartaruga vai percorrer.
# Cada traco vira uma linha; entre tracos a caneta sobe (nao desenha).
fig, ax = plt.subplots(figsize=(7, 7))
for t in tracos:
    xs = [pt[0] for pt in t]
    ys = [pt[1] for pt in t]
    ax.plot(xs, ys, 'k-', linewidth=1.2)
ax.set_xlim(0, 11); ax.set_ylim(0, 11); ax.set_aspect('equal')
ax.set_title(f'{len(tracos)} segmentos / {total} waypoints')
ax.grid(alpha=0.3)
plt.show()

## 6. Salvar e rodar no turtlesim

A célula abaixo roda a pipeline inteira e gera `output/contour.json`.

In [ ]:
# Executa as 5 etapas em sequencia e salva o JSON com os waypoints
p.executar(IMG, saida='output', largura=70, limiar_intensidade=0.45, limiar_borda=0.5)

### Como rodar no turtlesim

Dois terminais Ubuntu:

```bash
# Terminal 1
ros2 run turtlesim turtlesim_node

# Terminal 2
cd ~/ros2_ws && source install/setup.bash
ros2 run turtle_circle draw_node --ros-args -p arquivo_contorno:=$HOME/ros2_ws/src/turtle_circle/output/contour.json
```

O nó usa controle proporcional: para cada waypoint calcula a distância $\rho$ e o erro angular $\alpha$, publica `v_linear = K_p · ρ` e `v_angular = K_p · α` em `/turtle1/cmd_vel`. Entre segmentos, levanta a caneta com `/turtle1/set_pen`.